In [4]:
import operator
import random
import json
import logging
import os
from datetime import datetime
from deap import base, creator, tools, gp
import pydot

# --------------------------------------------------------------------------
# 1. 기본 설정 (지표, 타입, 조건 생성기)
# --------------------------------------------------------------------------
INDICATORS = ["RSI", "SMA", "MACD", "ATR", "BB"]
NUM_INDICATOR_PARAMS = [5, 10, 14, 20, 30, 50, 100, 150, 200]
SOURCES = ["open", "high", "low", "close", "volume"]
OPS = [operator.gt, operator.ge, operator.eq, operator.ne, operator.lt, operator.le]

class BuyType(object): pass
class SellType(object): pass

def Strategy(buy_logic, sell_logic):
    return (buy_logic, sell_logic)

class ConditionManager:
    def __init__(self, num_conditions=50):
        self.buy_pool = {}
        self.sell_pool = {}
        self.generate_conditions(num_conditions)
    
    def _create_random_condition(self):
        indicator, source, param = random.choice(INDICATORS), random.choice(SOURCES), random.choice(NUM_INDICATOR_PARAMS)
        op_func, value = random.choice(OPS), random.uniform(1, 100)
        return {"indicator": indicator, "source": source, "param": param, "op": op_func.__name__, "value": round(value, 2)}

    def generate_conditions(self, n):
        for i in range(n):
            self.buy_pool[f"buy_system{i + 1}"] = self._create_random_condition()
            self.sell_pool[f"sell_system{i + 1}"] = self._create_random_condition()

# --------------------------------------------------------------------------
# 2. DEAP 환경 설정
# --------------------------------------------------------------------------
condition_manager = ConditionManager()
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)
pset = gp.PrimitiveSetTyped("Main", [], object)
pset.addPrimitive(Strategy, [BuyType, SellType], object)
pset.addPrimitive(operator.and_, [BuyType, BuyType], BuyType); pset.addPrimitive(operator.or_, [BuyType, BuyType], BuyType); pset.addPrimitive(operator.not_, [BuyType], BuyType)
pset.addPrimitive(operator.and_, [SellType, SellType], SellType); pset.addPrimitive(operator.or_, [SellType, SellType], SellType); pset.addPrimitive(operator.not_, [SellType], SellType)
for term in condition_manager.buy_pool.keys(): pset.addTerminal(term, BuyType) 
for term in condition_manager.sell_pool.keys(): pset.addTerminal(term, SellType)

# --------------------------------------------------------------------------
# 3. 커스텀 교차 & 변이 함수
# --------------------------------------------------------------------------
def custom_crossover(ind1, ind2):
    if len(ind1) < 2 or len(ind2) < 2 or ind1[0].name != 'Strategy' or ind2[0].name != 'Strategy': return ind1, ind2
    if random.random() < 0.5:
        slice1, slice2 = ind1.searchSubtree(1), ind2.searchSubtree(1)
        sub_tree1, sub_tree2 = gp.PrimitiveTree(ind1[slice1]), gp.PrimitiveTree(ind2[slice2])
        gp.cxOnePoint(sub_tree1, sub_tree2)
        ind1[slice1], ind2[slice2] = sub_tree1, sub_tree2
    else:
        buy_slice1, buy_slice2 = ind1.searchSubtree(1), ind2.searchSubtree(1)
        sub_tree1, sub_tree2 = gp.PrimitiveTree(ind1[buy_slice1.stop:]), gp.PrimitiveTree(ind2[buy_slice2.stop:])
        gp.cxOnePoint(sub_tree1, sub_tree2)
        ind1[buy_slice1.stop:], ind2[buy_slice2.stop:] = sub_tree1, sub_tree2
    return ind1, ind2

def custom_mutation(individual):
    if len(individual) < 2 or individual[0].name != 'Strategy': return individual,
    subtree_slice = individual.searchSubtree(1) if random.random() < 0.5 else slice(individual.searchSubtree(1).stop, len(individual))
    selected_subtree_list = individual[subtree_slice]
    if not selected_subtree_list: return individual,
    temp_tree = gp.PrimitiveTree(selected_subtree_list)
    index = random.randrange(len(temp_tree))
    node = temp_tree[index]
    new_expr = gp.genHalfAndHalf(pset, 1, 4, type_=node.ret)
    temp_tree[temp_tree.searchSubtree(index)] = new_expr
    individual[subtree_slice] = temp_tree
    return individual,

# --------------------------------------------------------------------------
# 4. 평가, 파싱, 파일 저장 및 시각화 함수
# --------------------------------------------------------------------------
def eval_func(individual):
    if not individual or individual[0].name != 'Strategy': return -1000.0,
    if len(individual) > 100: return -100.0,
    return random.uniform(10, 100),

def parse_gp_tree_to_json(individual, condition_manager):
    if not isinstance(individual, gp.PrimitiveTree) or individual[0].name != 'Strategy': return None
    buy_subtree_slice = individual.searchSubtree(1)
    buy_tree, sell_tree = gp.PrimitiveTree(individual[buy_subtree_slice]), gp.PrimitiveTree(individual[buy_subtree_slice.stop:])
    buy_op_str, sell_op_str = (str(t) if len(t) > 0 else "NO_LOGIC" for t in (buy_tree, sell_tree))
    buy_systems_obj = {n.name: condition_manager.buy_pool[n.name] for n in buy_tree if isinstance(n, gp.Terminal)}
    sell_systems_obj = {n.name: condition_manager.sell_pool[n.name] for n in sell_tree if isinstance(n, gp.Terminal)}
    dynamic_vars = {f"{d['indicator']}_{d['param']}": d['param'] for d in {**buy_systems_obj, **sell_systems_obj}.values()}
    return {"vars": dynamic_vars, "buy_systems": buy_systems_obj, "buy_system_op": buy_op_str, "sell_systems": sell_systems_obj, "sell_system_op": sell_op_str}

def save_json_strategy(strategy_dict, output_dir):
    filepath = os.path.join(output_dir, "strategy.json")
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(strategy_dict, f, indent=4, ensure_ascii=False)
        logging.info(f"💾 JSON 전략 저장 완료: '{filepath}'")
    except Exception as e:
        logging.error(f"❌ JSON 파일 저장 실패: {e}")

def visualize_tree(individual, output_dir):
    filepath = os.path.join(output_dir, "strategy_tree.png")
    nodes, edges, labels = gp.graph(individual)
    graph = pydot.Dot(graph_type='graph', bgcolor="#f0f0f0") 
    
    for node_idx, node_label in labels.items():
        if node_label == 'Strategy': fillcolor = "#c5e3c5"
        elif node_label.startswith("buy_system"): fillcolor = "#f2c5c5"
        elif node_label.startswith("sell_system"): fillcolor = "#c5d5e3"
        elif node_label in ['and_', 'or_', 'not_']: fillcolor = "#e3d1c5"
        else: fillcolor = "#ffffff"
        graph.add_node(pydot.Node(node_idx, label=node_label, style="filled", fillcolor=fillcolor))

    for edge in edges: graph.add_edge(pydot.Edge(edge[0], edge[1]))
    try:
        graph.write_png(filepath)
        logging.info(f"📊 트리 시각화 완료: '{filepath}'")
    except OSError as e:
        logging.error(f"❌ 트리 시각화 실패: Graphviz를 찾을 수 없습니다. (오류: {e})")

def setup_logging():
    now = datetime.now()
    log_dir = os.path.join("logs", now.strftime("%Y-%m-%d"))
    run_dir_name = "run_" + now.strftime("%H-%M-%S")
    output_dir = os.path.join(log_dir, run_dir_name)
    os.makedirs(output_dir, exist_ok=True)
    
    log_filepath = os.path.join(output_dir, "strategy.log")
    
    # ✨ --- [핵심 수정] force=True 인자 추가 --- ✨
    # 이 옵션은 다른 라이브러리에 의해 생성된 기존 로깅 설정을 제거하고
    # 현재 설정을 강제로 적용하여 파일 로깅이 누락되는 것을 방지합니다.
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filepath, encoding='utf-8'), # UTF-8 인코딩 추가
            logging.StreamHandler()
        ],
        force=True 
    )
    logging.info(f"결과가 '{output_dir}' 폴더에 저장됩니다.")
    return output_dir

# --------------------------------------------------------------------------
# 5. Toolbox 설정 및 메인 실행 블록
# --------------------------------------------------------------------------
toolbox = base.Toolbox()
toolbox.register("expr_init", gp.genHalfAndHalf, pset=pset, min_=2, max_=6, type_=object)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr_init)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", eval_func)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", custom_crossover)
toolbox.register("mutate", custom_mutation)

if __name__ == "__main__":
    output_dir = setup_logging()
    random.seed(42)
    
    N_POP, N_GEN, CXPB, MUTPB = 100, 20, 0.8, 0.15
    pop = toolbox.population(n=N_POP)
    
    logging.info("✨ 초기 개체군 생성 및 평가 시작...")
    fitnesses = map(toolbox.evaluate, pop)
    for ind, fit in zip(pop, fitnesses): ind.fitness.values = fit
    
    logging.info("="*60)
    logging.info(f"🚀 유전 프로그래밍 진화 시작! (세대: {N_GEN}, 개체수: {N_POP})")
    logging.info("="*60)

    for g in range(N_GEN):
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values, child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses): ind.fitness.values = fit
        pop[:] = offspring
        fits = [ind.fitness.values[0] for ind in pop]
        logging.info(f"> 세대 {g+1:02d}: 최고={max(fits):.2f}, 평균={sum(fits)/len(pop):.2f}")

    logging.info("="*60)
    logging.info("🏆 최종 최적 전략 탐색 완료!")
    
    best_ind = tools.selBest(pop, 1)[0]
    final_json_strategy = parse_gp_tree_to_json(best_ind, condition_manager)
    
    logging.info(f"최적 개체 트리 구조 (크기: {len(best_ind)}):\n{str(best_ind)}")
    logging.info(f"최고 적합도: {best_ind.fitness.values[0]:.2f}")
    
    if final_json_strategy:
        json_output = json.dumps(final_json_strategy, indent=4, ensure_ascii=False)
        save_json_strategy(final_json_strategy, output_dir)
        visualize_tree(best_ind, output_dir)
    else:
        logging.error("❌ 최종 JSON 생성 실패: 최적 개체가 유효한 'Strategy' 구조가 아닙니다.")

/home/tako/Documents/yonghan/GenTradingRuleWithGP/.conda/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/tako/Documents/yonghan/GenTradingRuleWithGP/.conda/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
2025-09-28 00:36:45,911 [INFO] 결과가 'logs/2025-09-28/run_00-36-45' 폴더에 저장됩니다.
2025-09-28 00:36:45,914 [INFO] ✨ 초기 개체군 생성 및 평가 시작...
2025-09-28 00:36:45,915 [INFO] ============================================================
2025-09-28 00:36:45,916 [INFO] 🚀 유전 프로그래밍 진화 시작! (세대: 20, 개체수: 100)
2025-09-28 00:36:45,917 [INFO] ==